# MedHopQA: tiny public-data and evaluation tutorial

This notebook accompanies [MedHopQA (arXiv:2605.12361)](https://arxiv.org/abs/2605.12361). It loads the 15 examples published on the [official NLM page](https://www.ncbi.nlm.nih.gov/research/bionlp/medhopqa), represents them as JSON, and demonstrates a lightweight version of the benchmark's synonym-aware lexical evaluation.

> **Availability:** CodaBench distributes 10,000 questions, with the 1,000 scored questions concealed inside that pool. Their answers are withheld by design. This notebook uses only the 15 answers that the authors publicly display.

> **Safety:** This is an educational QA benchmark demo, not medical advice or clinical decision support.

## 1. Load the official public examples

The project page is ordinary HTML, so Python's standard library is enough. We extract visible text and then locate every `Question: ... Answer: ...` pair. Stable local IDs are added because the web page does not publish record IDs.

In [ ]:
import json
import re
import unicodedata
import urllib.request
from html.parser import HTMLParser

PUBLIC_PAGE = 'https://www.ncbi.nlm.nih.gov/research/bionlp/medhopqa'

# These are copied from the same official page and are used only if networking fails.
OFFLINE_PUBLIC_FALLBACK = [
    {
        'id': 'official_example_01',
        'question': ('Which benign, autosomal recessive hyperbilirubinemia syndrome '
                     'is caused by a mutation in the UGT1A1 gene on chromosome 2?'),
        'answer': 'Gilbert syndrome',
        'source': 'NLM MedHopQA public examples (offline copy)',
    },
    {
        'id': 'official_example_02',
        'question': ('Is the craniosynostosis seen in Shprintzen–Goldberg syndrome '
                     'caused by a mutation in the fibrillin-1 gene?'),
        'answer': 'No',
        'source': 'NLM MedHopQA public examples (offline copy)',
    },
]

class VisibleText(HTMLParser):
    def __init__(self):
        super().__init__()
        self.parts = []

    def handle_data(self, data):
        cleaned = ' '.join(data.split())
        if cleaned:
            self.parts.append(cleaned)

def load_public_examples(url=PUBLIC_PAGE):
    request = urllib.request.Request(url, headers={'User-Agent': 'MedHopQA-Colab-Tutorial/1.0'})
    try:
        with urllib.request.urlopen(request, timeout=30) as response:
            html = response.read().decode('utf-8')
        parser = VisibleText()
        parser.feed(html)
        page_text = ' '.join(parser.parts)
        block = page_text.split('These are 15 example questions and their answers:', 1)[1]
        block = block.split('Please visit', 1)[0]
        pairs = re.findall(
            r'Question:\s*(.*?)\s*Answer:\s*(.*?)(?=\s*Question:|$)', block
        )
        if len(pairs) != 15:
            raise ValueError(f'Expected 15 public examples, found {len(pairs)}')
        return [
            {
                'id': f'official_example_{i:02d}',
                'question': question.strip(),
                'answer': answer.strip(),
                'source': 'NLM MedHopQA public examples',
            }
            for i, (question, answer) in enumerate(pairs, start=1)
        ], 'live official page'
    except Exception as error:
        print(f'Official page could not be loaded ({error}). Using 2 public offline examples.')
        return OFFLINE_PUBLIC_FALLBACK, 'offline public fallback'

examples, load_mode = load_public_examples()
print(f'Loaded {len(examples)} examples from: {load_mode}')
print(json.dumps(examples[:2], indent=2, ensure_ascii=False))

## 2. What makes a question multi-hop?

A single passage may not settle a MedHopQA question. The intended pattern is: retrieve one article, identify a connecting entity, retrieve a second article, and combine the facts. Below is a **simplified teaching trace** for public example 2. It illustrates the structure; it is not an automatically retrieved evidence annotation from the dataset.

In [ ]:
teaching_trace = {
    'question': OFFLINE_PUBLIC_FALLBACK[1]['question'],
    'hop_1': {
        'article_topic': 'Shprintzen–Goldberg syndrome',
        'use': 'Identify the gene associated with the syndrome and its craniosynostosis phenotype.',
    },
    'hop_2': {
        'article_topic': 'Fibrillin-1',
        'use': 'Check whether that proposed gene/protein is the causal match.',
    },
    'composition': 'Compare the entity from hop 1 with the entity proposed in hop 2.',
    'public_answer': 'No',
    'note': 'Illustrative reasoning structure; not an official evidence annotation.',
}
print(json.dumps(teaching_trace, indent=2, ensure_ascii=False))

## 3. Normalized and synonym-aware lexical matching

Plain exact match is brittle: `Bisphenol A`, `bisphenol-a`, and `BPA` may denote the same answer. The paper first compares normalized predictions with the canonical answer and curated variants. Its full evaluator also uses answer-type-aware concept validation and MedCPT; here we implement only the transparent lexical layer.

In [ ]:
def normalize_answer(text):
    text = unicodedata.normalize('NFKC', str(text)).casefold()
    text = re.sub(r'[^\w]+', ' ', text, flags=re.UNICODE)
    return ' '.join(text.split())

def lexical_match(prediction, canonical_answer, accepted_variants=()):
    accepted = {normalize_answer(canonical_answer)}
    accepted.update(normalize_answer(item) for item in accepted_variants)
    return normalize_answer(prediction) in accepted

checks = [
    {'prediction': 'gilbert syndrome', 'gold': 'Gilbert syndrome', 'variants': []},
    {'prediction': 'BPA', 'gold': 'Bisphenol A', 'variants': ['BPA', 'Bisphenol-A']},
    {'prediction': 'fibrillin 1', 'gold': 'Fibrillin-1', 'variants': ['fibrillin 1']},
    {'prediction': 'Marfan syndrome', 'gold': 'Gilbert syndrome', 'variants': []},
]

for row in checks:
    row['matched'] = lexical_match(row['prediction'], row['gold'], row['variants'])
    print(row)

The alias list is part of the evaluation data, not something a system should invent after seeing its score. This tiny demo includes only illustrative aliases. Reproducing the paper's full evaluator requires its curated lexicons, answer-type routing, and concept-level semantic checks.

## 4. Score a few public-example predictions

For this tutorial we provide predictions only for examples whose official answers appear on the public page. Unanswered rows are skipped rather than treated as wrong.

In [ ]:
predictions = {
    'official_example_01': 'gilbert syndrome',
    'official_example_02': 'No',
}

scored = []
for item in examples:
    if item['id'] not in predictions:
        continue
    prediction = predictions[item['id']]
    is_correct = lexical_match(prediction, item['answer'])
    scored.append({
        'id': item['id'],
        'prediction': prediction,
        'gold': item['answer'],
        'correct': is_correct,
    })

accuracy = sum(row['correct'] for row in scored) / len(scored)
print(json.dumps(scored, indent=2, ensure_ascii=False))
print(f'Normalized exact-match accuracy: {accuracy:.1%}')

## 5. Prediction JSON for the protected benchmark

A real system should emit predictions without expecting access to the hidden answer key. The object below is a **local teaching schema**, not the official CodaBench upload format; always follow the competition page's current submission instructions.

In [ ]:
prediction_record = {
    'question_id': 'hidden_or_unlabeled_item_id',
    'prediction': 'model answer goes here',
}
print(json.dumps(prediction_record, indent=2))

## What this notebook does—and does not do

You now have the full small loop: load public examples → inspect JSON → understand the two-hop structure → normalize predictions → score public labels → emit prediction-only records.

It does **not** reveal the protected 1,000-answer key, reproduce expert curation, retrieve the paired Wikipedia articles, or implement the paper's MedCPT-based concept validation. Those are deliberately beyond this minimal, CPU-only tutorial.